# CottonLens — ücretsiz veri araştırması v1
Eski A turu yeniden çalışmaz. Yeni ve ayrı deney: research-free-data-v1.
Kurulum, kaynak erişimi ve prepare hücrelerini sırayla çalıştır; son hücrede STAGE seç. Varsayılan status eğitim yapmaz.
diagnose: küçük GPU kontrolleri; ablate: aynı originlerde 4 feature paketi (uzun sürebilir); search: yalnız incelenmiş yeni yayın paketleri ve tamamlanmış ablation sonrası Optuna.
Uydu, TFT ve schema-v4 API belirsizlik release desteği bu notebook sürümünde henüz yok.
Hata veya kesinti halinde aynı kaynak/deney kimliğiyle devam et. Eski writer lock dosyasını otomatik silme.


In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/CottonLensAI')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
EXPERIMENT = 'research-free-data-v1'  # Retain this identifier for resumes of EXACTLY this data/source.


In [ ]:
import hashlib, importlib, json, sys, zipfile
from pathlib import PurePosixPath
SOURCE_ZIP = DRIVE_ROOT / 'sources/cottonlens-free-research-source-v1.zip'
expected_zip = SOURCE_ZIP.with_suffix('.zip.sha256').read_text().split()[0]
assert hashlib.sha256(SOURCE_ZIP.read_bytes()).hexdigest() == expected_zip, 'Source ZIP checksum mismatch'
with zipfile.ZipFile(SOURCE_ZIP) as archive:
    source = json.loads(archive.read('.source-manifest.json'))
    payload = {key: value for key, value in source.items() if key != 'source_id'}
    identity = hashlib.sha256(json.dumps(payload, sort_keys=True, separators=(',', ':')).encode()).hexdigest()
    assert identity == source['source_id'], 'Source manifest mismatch'
    assert len(archive.namelist()) == len(set(archive.namelist())), 'Duplicate archive members'
    assert set(archive.namelist()) == set(source['files']) | {'.source-manifest.json'}, 'Unexpected archive files'
    REPO = Path('/content') / ('cottonlens-source-' + identity[:16])
    REPO.mkdir(exist_ok=True)
    for name in archive.namelist():
        member = PurePosixPath(name)
        assert not member.is_absolute() and '..' not in member.parts and chr(92) not in name and ':' not in name
        content = archive.read(name)
        if name in source['files']:
            assert hashlib.sha256(content).hexdigest() == source['files'][name], name
        target = REPO / name
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.exists():
            assert target.read_bytes() == content, f'Existing Colab source changed: {name}'
        else:
            with target.open('xb') as handle:
                handle.write(content)
sys.path.insert(0, str(REPO / 'ml'))
import colab_setup
importlib.reload(colab_setup)
colab_setup.PROJECT = REPO
from colab_setup import setup, run, INFERENCE
print('Source identity:', identity, 'Git HEAD:', source['git_head'], 'Includes working-tree edits:', source['working_tree_dirty'])


In [ ]:
PYTHON = setup(research=True)  # Locked managed Python 3.12.11; no global packages.
print('Training interpreter:', PYTHON)
print('Inference-only interpreter:', INFERENCE / 'bin/python')


In [ ]:
# Free USDA keys: add these names in Colab's left sidebar -> Secrets.
# Never paste credential values into this notebook.
from google.colab import userdata
import os
SOURCE_ACTION = 'status'  # status, catalog, nass
SOURCE_YEAR = 2020
key_names = ('USDA_AMS_API_KEY', 'USDA_FAS_API_KEY', 'USDA_NASS_API_KEY')
available = []
for name in key_names:
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if value:
        os.environ[name] = value
        available.append(name)
    else:
        os.environ.pop(name, None)
print('Available credential names:', available)
print('Missing credential names:', [name for name in key_names if name not in available])
print('FAS signup: https://api.data.gov/signup/')
print('NASS signup: https://quickstats.nass.usda.gov/api/')
print('AMS guide: https://mymarketnews.ams.usda.gov/mymarketnews-api/authentication')
RAW_USDA = DRIVE_ROOT / 'data' / 'free-usda-raw'
if SOURCE_ACTION == 'catalog':
    for provider, name in (('ams', 'USDA_AMS_API_KEY'), ('fas', 'USDA_FAS_API_KEY')):
        if name in available:
            run([PYTHON, '-m', 'cottonlens_ml.sources.usda', '--provider', provider, '--output', RAW_USDA])
        else:
            print(provider, 'catalog skipped: add its free key first')
elif SOURCE_ACTION == 'nass':
    assert 'USDA_NASS_API_KEY' in available, 'Add USDA_NASS_API_KEY in Colab Secrets'
    run([PYTHON, '-m', 'cottonlens_ml.sources.usda', '--provider', 'nass', '--year', str(SOURCE_YEAR), '--output', RAW_USDA])
else:
    assert SOURCE_ACTION == 'status', 'Choose status, catalog or nass'


In [ ]:
# Add only reviewed immutable publication packages before the FIRST prepare.
# Adding/changing packages later requires a NEW EXPERIMENT name.
PUBLICATION_PACKAGES = []
BASE = [PYTHON, '-m', 'cottonlens_ml.research.engine', '--repo', REPO,
        '--drive-root', DRIVE_ROOT, '--experiment', EXPERIMENT]
package_args = [argument for path in PUBLICATION_PACKAGES for argument in ('--publication-package', str(path))]
run([*BASE, '--stage', 'prepare', '--profile', 'free-data-v1', *package_args])
run([*BASE, '--stage', 'status'])


In [ ]:
ROUND = 'A'  # A: adaptive tabular, B: direction, C: sequence, E: history/ensemble
EXTENSION = 0  # A only; 1 adds 128 candidates when previous inner evidence permits
STAGE = 'status'  # diagnose, ablate, search, report; lock/reproduce/export after evidence review
assert STAGE in ('status', 'diagnose', 'ablate', 'search', 'report', 'lock', 'reproduce', 'export', 'track')
import colab_progress
from IPython.display import display, Markdown
from colab_setup import process_env
folder = DRIVE_ROOT / 'experiments' / EXPERIMENT
ready = json.loads((folder / 'ready.json').read_text())
fold_count = len(ready['identity']['split']['folds'])
group_count = len(ready['identity']['groups'])
namespace = 'ablate-*/*.json' if STAGE == 'ablate' else (('adaptive-A' if EXTENSION == 0 else f'adaptive-A-extension-{EXTENSION}') + '/*.json' if ROUND == 'A' else f'free-{ROUND}*/*.json')
units_per_fold = group_count * 4 if STAGE == 'ablate' else {'A': 4, 'B': 6, 'C': 10, 'E': 4}[ROUND]
total = units_per_fold * fold_count
print('Maximum decision units (skipped E units may leave the bar incomplete; not elapsed time):', total)
if STAGE == 'status':
    run([*BASE, '--stage', STAGE])
else:
    panel = display(Markdown('Starting stage; completed fits will be restored.'), display_id=True)
    def update(message):
        panel.update(Markdown('```text\n' + message + '\n```'))
    colab_progress.supervise([*BASE, '--stage', STAGE, '--round', ROUND, '--extension', str(EXTENSION)], cwd=REPO, env=process_env(),
        drive_logs=DRIVE_ROOT / 'reports', decisions=folder / 'outer',
        decision_pattern=namespace, total=total, fold_count=fold_count, update=update)
